In [ ]:
import sys

sys.path.append("..")

In [ ]:
from data.dataloader import LowLightDataModule
from model.blocks.homomorphic import ImageDecomposition, ImageComposition
from model.blocks.illuminationenhancer import IlluminationEnhancer
from utils.utils import show_batch, summarize_model

In [ ]:
data_module = LowLightDataModule(
    train_dir="../data/1_train",
    valid_dir="../data/2_valid",
    bench_dir="../data/3_bench",
    infer_dir="../data/4_infer",
    image_size=512,
    batch_size=1,
    num_workers=4,
)

data_module.setup(stage="fit")

In [ ]:
train_dataloader = data_module.train_dataloader()

In [ ]:
data = next(iter(train_dataloader))
print(data.shape)
show_batch(images=data)

In [ ]:
decompose = ImageDecomposition(offset=0.5, cutoff=0.1)
compose = ImageComposition(offset=0.5)

In [ ]:
data = data.cuda()
decompose = decompose.cuda()
compose = compose.cuda()

In [ ]:
luminance, chroma_red, chroma_blue, illuminance, reflectance = decompose(data)
rgb, lu = compose(chroma_red, chroma_blue, illuminance, reflectance)

In [ ]:
unet = IlluminationEnhancer(
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    num_resolution=4,
    dropout_ratio=0.2,
)

In [ ]:
unet = unet.cuda()

In [ ]:
enhanced = unet(illuminance)

In [ ]:
show_batch(images=enhanced)

In [ ]:
summarize_model(model=unet, input_size=(1, 1, 256, 256))